# 01 — Web Scraping

Collect product data and save it as CSV.

In [1]:
%pip install requests scrapy pandas matplotlib mysql-connector-python selenium

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hemam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import time
import re
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scrapy import Selector
from IPython.display import display
from mysql.connector import connect, Error


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json
import time

# ============================================================
# 1. SHOPSY KITCHEN CONTAINERS URL
# ============================================================

SHOPSY_URL = "https://www.shopsy.in/household/containers-bottles/containers-jars/kitchen-containers/pr?sid=r4l%2Cv2a%2C29e%2C4cf"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/151.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

# ============================================================
# 2. GET LISTING PAGE
# ============================================================

response = requests.get(
    SHOPSY_URL,
    headers=headers,
    timeout=30
)
response.raise_for_status()
print("Listing Status:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

# ============================================================
# 3. COLLECT PRODUCT LINKS
# ============================================================

product_links = []
for anchor in soup.find_all("a", href=True):
    href = anchor["href"]
    if "/p/" in href:
        if href.startswith("/"):
            href = "https://www.shopsy.in" + href
        if href not in product_links:
            product_links.append(href)

product_links = list(dict.fromkeys(product_links))[:30]
print("Product links found:", len(product_links))

Listing Status: 200
Product links found: 38
Scraping 1/30
Scraping 2/30
Scraping 3/30
Scraping 4/30
Scraping 5/30
Scraping 6/30
Scraping 7/30
Scraping 8/30
Scraping 9/30
Scraping 10/30
Scraping 11/30
Scraping 12/30
Scraping 13/30
Scraping 14/30
Scraping 15/30
Scraping 16/30
Scraping 17/30
Scraping 18/30
Scraping 19/30
Scraping 20/30
Scraping 21/30
Scraping 22/30
Scraping 23/30
Scraping 24/30
Scraping 25/30
Scraping 26/30
Scraping 27/30
Scraping 28/30
Scraping 29/30
Scraping 30/30

SCRAPING COMPLETED
Products scraped: 30
CSV file: shopsy_kitchen_products.csv


,Product_Name,Price,Discount,Rating,Reviews,Brand,Capacity,Material,Product_URL
0,VASOYA Pack of 1 Plastic Fridge Container - 27...,₹161,86%,4,25,Shopsy,"2700 ml, 1500 ml, 500 ml",Plastic,https://www.shopsy.in/vasoya-pack-1-plastic-fr...
1,AneriDEALS Pack of 24 Plastic Grocery Containe...,₹423,78%,4.1,34,Shopsy,"250 ml, 350 ml, 650 ml, 1200 ml, 1000 ml",Plastic,https://www.shopsy.in/anerideals-pack-24-plast...
2,BELIZZI Pack of 6 Plastic Fridge Container - 1...,₹257,74%,4.1,272,Shopsy,"1500 ml, 500 ml, 1000 ml",Plastic,https://www.shopsy.in/belizzi-pack-6-plastic-f...
3,KIKANII Pack of 12 Plastic Grocery Container -...,₹351,64%,3.4,,Shopsy,500 ml,Plastic,https://www.shopsy.in/kikanii-pack-12-plastic-...
4,COSY HOME MAKE IT EASY Pack of 1 Plastic Egg C...,₹158,80%,3.8,12,Shopsy,,"Plastic, Silicone, Wood",https://www.shopsy.in/cosy-home-make-easy-pack...
5,VR Pack of 3 Plastic Grocery Container - 4500 ...,₹247,75%,4.1,210,Shopsy,"4500 ml, 250 ml",Plastic,https://www.shopsy.in/vr-pack-3-plastic-grocer...
6,Veksin Pack of 6 Plastic Grocery Container - 1...,₹473,66%,4.3,23,Shopsy,"1000 ml, 2000 ml, 3000 ml, 6000 ml, 8000 ml, 1...",Plastic,https://www.shopsy.in/veksin-pack-6-plastic-gr...
7,Panel's Pack of 6 Glass Grocery Container - 30...,₹330,66%,4.5,27,Shopsy,"300 ml, 300ml","Stainless Steel, Glass, Steel",https://www.shopsy.in/panel-s-pack-6-glass-gro...
8,gptrade Pack of 3 Steel Tea Coffee & Sugar Con...,₹267,66%,4,9,Shopsy,"500 ml, 300 ml, 200 ml","Stainless Steel, Steel",https://www.shopsy.in/gptrade-pack-3-steel-tea...
9,Finner Pack of 8 Plastic Grocery Container - 1...,₹198,66%,3.7,1,Shopsy,100 ml,Plastic,https://www.shopsy.in/finner-pack-8-plastic-gr...


In [ ]:
# ============================================================
# 5. CREATE DATAFRAME AND SAVE CSV
# ============================================================

df = pd.DataFrame(data).drop_duplicates(subset=["Product_URL"])
output_file = "shopsy_kitchen_products.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\n" + "=" * 60)
print("SCRAPING COMPLETED")
print("=" * 60)
print("Products scraped:", len(df))
print("CSV file:", output_file)
display(df.head(20))

In [ ]:
# ============================================================
# 4. SCRAPE EACH PRODUCT PAGE
# ============================================================

data = []

for index, product_url in enumerate(product_links, start=1):
    print(f"Scraping {index}/{len(product_links)}")

    try:
        product_response = requests.get(
            product_url,
            headers=headers,
            timeout=30
        )
        product_response.raise_for_status()
        product_soup = BeautifulSoup(product_response.text, "html.parser")
        page_text = clean(product_soup.get_text(" ", strip=True))

        jsonld_data = []
        for script in product_soup.find_all("script", type="application/ld+json"):
            try:
                value = json.loads(script.string or script.get_text())
                jsonld_data.extend(value if isinstance(value, list) else [value])
            except (TypeError, json.JSONDecodeError):
                continue

        product_name = ""
        price = ""
        rating = ""
        reviews = ""
        brand = ""

        for item in jsonld_data:
            if not isinstance(item, dict) or item.get("@type") != "Product":
                continue

            product_name = clean(item.get("name", ""))
            brand_data = item.get("brand")
            if isinstance(brand_data, dict):
                brand = clean(brand_data.get("name", ""))
            elif isinstance(brand_data, str):
                brand = clean(brand_data)

            rating_data = item.get("aggregateRating", {})
            if isinstance(rating_data, dict):
                rating = clean(rating_data.get("ratingValue", ""))
                reviews = clean(rating_data.get("reviewCount", rating_data.get("ratingCount", "")))

            offers = item.get("offers", {})
            if isinstance(offers, dict) and offers.get("price"):
                price = "₹" + str(offers["price"])
            break

        if not product_name and product_soup.title:
            product_name = re.sub(
                r"\s*[-|]\s*Shopsy.*$", "",
                clean(product_soup.title.get_text()),
                flags=re.I
            )

        if not price:
            price = find_price(page_text)
        discount = find_discount(page_text)
        rating = rating or find_rating(page_text)
        reviews = reviews or find_reviews(page_text)

        if not brand:
            brand_match = re.search(
                r"Brand\s*[:\-]?\s*([A-Za-z0-9 &._-]+)",
                page_text,
                re.I
            )
            if brand_match:
                brand = clean(brand_match.group(1))

        capacity = find_capacity(page_text) or find_capacity(product_name)
        material = find_material(page_text) or find_material(product_name)

        data.append({
            "Product_Name": product_name,
            "Price": price,
            "Discount": discount,
            "Rating": rating,
            "Reviews": reviews,
            "Brand": brand,
            "Capacity": capacity,
            "Material": material,
            "Product_URL": product_url
        })
        time.sleep(1)

    except requests.RequestException as error:
        print("Request error:", str(error)[:100])
    except Exception as error:
        print("Error:", str(error)[:100])

In [ ]:
def clean(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def find_price(text):
    for pattern in [r"₹\s*([\d,]+)", r"Rs\.?\s*([\d,]+)"]:
        match = re.search(pattern, text, re.I)
        if match:
            return "₹" + match.group(1)
    return ""


def find_discount(text):
    for pattern in [r"(\d{1,3})\s*%\s*off", r"(\d{1,3})\s*%\s*discount"]:
        match = re.search(pattern, text, re.I)
        if match:
            return match.group(1) + "%"
    return ""


def find_rating(text):
    for pattern in [r"([1-5]\.\d)\s*(?:★|Stars?|Ratings?)", r"\b([1-5]\.\d)\b"]:
        match = re.search(pattern, text, re.I)
        if match:
            value = float(match.group(1))
            if 1 <= value <= 5:
                return match.group(1)
    return ""


def find_reviews(text):
    for pattern in [r"([\d,]+)\s*Reviews?", r"([\d,]+)\s*Ratings?"]:
        match = re.search(pattern, text, re.I)
        if match:
            return match.group(1)
    return ""


def find_capacity(text):
    patterns = [
        r"\b\d+(?:\.\d+)?\s*(?:ml|ML)\b",
        r"\b\d+(?:\.\d+)?\s*(?:l|L)\b",
        r"\b\d+(?:\.\d+)?\s*(?:litre|litres)\b"
    ]
    values = []
    for pattern in patterns:
        values.extend(re.findall(pattern, text, re.I))
    return ", ".join(dict.fromkeys(values))


def find_material(text):
    materials = [
        "Stainless Steel", "Plastic", "Glass", "Steel", "Acrylic",
        "Ceramic", "Silicone", "Aluminium", "Aluminum", "Wood"
    ]
    found = [
        material for material in materials
        if re.search(r"\b" + re.escape(material) + r"\b", text, re.I)
    ]
    return ", ".join(dict.fromkeys(found))